## Hello World Example with A2A Server Setup

In [ ]:
%pip install a2a-sdk==0.3.8 azure-ai-projects python-dotenv azure-identity uvicorn a2a-sdk[http-server]==0.3.8

In [2]:
import importlib.metadata
import starlette
import sse_starlette

print("a2a-sdk:", importlib.metadata.version("a2a-sdk"))
print("Starlette:", starlette.__version__)
print("sse-starlette installerad.")

a2a-sdk: 0.3.8
Starlette: 1.7.0
sse-starlette installerad.


### Creating the Hello World Agent Class with Function Invocations

In [5]:
class HelloWorldAgent():
    """This class will contain helper functions for interacting with our Foundry Agent"""

    async def invoke_agent(self, user_query: str) -> str:
        return f"Hello, you said: {user_query}"



### Creating the A2A Agent Executor

In [6]:
from a2a.server.agent_execution import AgentExecutor, RequestContext
from a2a.server.events import EventQueue
from a2a.utils import new_agent_text_message
from a2a.types import (
    TaskArtifactUpdateEvent,
    TaskState,
    TaskStatus,
    TaskStatusUpdateEvent,
)
from a2a.utils import new_text_artifact


class HelloWorldAgentExecutor(AgentExecutor):
    """Hello World Agent Executor Definition."""

    def __init__(self):
        self.agent = HelloWorldAgent()
    async def execute(
        self,
        context: RequestContext,
        event_queue: EventQueue,
    ) -> None:
        query = context.get_user_input()
        if not context.message:
            raise Exception('No message provided')

        # If your agent does not support streaming, just call invoke_agent
        result = await self.agent.invoke_agent(query)

        message = TaskArtifactUpdateEvent(
            context_id=context.context_id,  # type: ignore
            task_id=context.task_id,  # type: ignore
            artifact=new_text_artifact(
                name='current_result',
                text=result,
            ),
        )
        await event_queue.enqueue_event(message)

        status = TaskStatusUpdateEvent(
            context_id=context.context_id,  # type: ignore
            task_id=context.task_id,  # type: ignore
            status=TaskStatus(state=TaskState.completed),
            final=True,
        )
        await event_queue.enqueue_event(status)

    async def cancel(
        self, context: RequestContext, event_queue: EventQueue
    ) -> None:
        raise Exception('cancel not supported')

### Creating the Agent Skill Definition

In [7]:
from a2a.types import (
    AgentCapabilities,
    AgentCard,
    AgentSkill,
)

skill = AgentSkill(
    id = "hello_world_agent_skill",
    name = "Responses API from Hello World Agent",
    description = "Responses API from Hello World Agent",
    tags = ["hello world agent"],
    examples = ["hi, how are you?", "can you tell me something about GenAI and LLMs"]
)

### Creating the Agent Card

In [8]:
public_agent_card = AgentCard(
    name = "Hello World Demo Agent",
    description = "Hello World Demo Agent to Show A2A Usage with Microsoft Foundry",
    url = "http://localhost:8080",
    version = "1.0.0",
    default_input_modes=['text'],
    default_output_modes=['text'],
    capabilities=AgentCapabilities(streaming=False),
    skills = [skill]
)

### Creating the Request Handler

In [9]:
from a2a.server.request_handlers import DefaultRequestHandler
from a2a.server.tasks import InMemoryTaskStore

request_handler = DefaultRequestHandler(
    agent_executor = HelloWorldAgentExecutor(),
    task_store = InMemoryTaskStore()
)

### Creating the A2A Server

In [10]:
from a2a.server.apps import A2AStarletteApplication

server = A2AStarletteApplication(
    agent_card=public_agent_card,
    http_handler=request_handler,
)

### Starting the A2A Server

Navigate to http://localhost:8080/.well-known/agent.json to see the agent public card

In [11]:
import asyncio
import uvicorn

config = uvicorn.Config(
    server.build(),
    host="0.0.0.0",
    port=8080,
    loop="asyncio",
)

server_instance = uvicorn.Server(config)

await server_instance.serve()

INFO:     Started server process [32940]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8080 (Press CTRL+C to quit)


INFO:     127.0.0.1:53698 - "GET /wellknown/agent.json HTTP/1.1" 404 Not Found
INFO:     127.0.0.1:53698 - "GET /favicon.ico HTTP/1.1" 404 Not Found
INFO:     127.0.0.1:49674 - "GET /well-known/agent.json HTTP/1.1" 404 Not Found


Deprecated agent card endpoint '/.well-known/agent.json' accessed. Please use '/.well-known/agent-card.json' instead. This endpoint will be removed in a future version.


INFO:     127.0.0.1:54583 - "GET /.well-known/agent.json HTTP/1.1" 200 OK
INFO:     127.0.0.1:54271 - "GET /.well-known/agent-card.json HTTP/1.1" 200 OK
INFO:     127.0.0.1:54271 - "POST / HTTP/1.1" 200 OK


INFO:     Shutting down
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.
INFO:     Finished server process [32940]
